# Lab 1 - RAG with ChromaDB (notebook version)

Runs top to bottom in **VS Code** or **Jupyter**. Same pipeline as the Streamlit lab:
PDF -> split -> embed -> vector store -> retrieve -> grounded answer.

**Setup:** keep this notebook in the same folder as `sandbox.py`, `grader.py`, and
`content/_data/sample_knowledge_base.pdf` (the Lab 1 bundle).

| Mode | Answer LLM | Embeddings | Needs |
|---|---|---|---|
| `mock` (default) | local context-grounded stand-in | DeterministicFakeEmbedding / FastEmbed | nothing |
| `openai` | ChatOpenAI | OpenAIEmbeddings | `OPENAI_API_KEY` |
| `claude` | ChatAnthropic | local | `ANTHROPIC_API_KEY` |


## 0 - Environment setup


In [ ]:
import sys, os, pathlib

# folder that holds sandbox.py / grader.py / content/  (edit if the notebook lives elsewhere)
HERE = pathlib.Path.cwd()
sys.path.insert(0, str(HERE))

# choose the mode BEFORE importing sandbox
os.environ["SANDBOX_MODE"] = "mock"          # "mock" | "openai" | "claude"
# os.environ["OPENAI_API_KEY"] = "sk-..."    # uncomment for openai mode
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # uncomment for claude mode

import sandbox
from sandbox import (get_chat_model, get_embeddings, make_vectorstore,
                     load_sample_docs, split_docs, SAMPLE_PDF)

MODE = os.environ["SANDBOX_MODE"]   # use exactly the mode you set above
print("MODE      =", MODE)
print("SAMPLE_PDF =", SAMPLE_PDF, "  exists:", pathlib.Path(SAMPLE_PDF).is_file())


## Task 1 - Load the PDF  (5 marks)

Load the knowledge base with LangChain's PDF loader; print the page count and first-page metadata.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(SAMPLE_PDF)
documents = loader.load()

print("Number of pages:", len(documents))
print("First page metadata:", documents[0].metadata)
print(documents[0].page_content[:200])


## Task 2 - Split into chunks  (20 marks)

`chunk_size=1000`, `chunk_overlap=200`.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 1000
chunk_overlap = 200
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunks = splitter.split_documents(documents)
print("Number of chunks:", len(chunks))


## Task 3 - Embeddings + vector store  (25 marks)

`get_embeddings(MODE)` + `make_vectorstore(chunks, embeddings)` (real ChromaDB if installed,
else `InMemoryVectorStore` - same interface). Then a similarity search.


In [ ]:
embeddings = get_embeddings(MODE)
vectorstore = make_vectorstore(chunks, embeddings)

query = "How many casual leave days are provided each year?"
results = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(results):
    print(f"--- match {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:200])


## Task 4 - Grounded question-answering  (25 marks)

Retriever (k=3) + `get_chat_model(MODE)` + a prompt that answers **only** from context,
else replies exactly `I could not find the answer in the provided document.`


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = get_chat_model(MODE)

prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below. Do not use outside knowledge.
If the answer is not in the context, reply exactly:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:"""
)

question = "What is the standard notice period for regular full-time employees?"
docs = retriever.invoke(question)
context = "\n\n".join(d.page_content for d in docs)
resp = llm.invoke(prompt.format_messages(context=context, question=question))

print("Answer:", resp.content)
print("Pages:", sorted({d.metadata.get("page", "?") for d in docs}))


## Task 5 - Interactive RAG app  (25 marks)

`answer()` wraps Task 4. The Streamlit lab uses `input()` in a `while True` loop;
in a notebook it's cleaner to call `answer()` directly or loop over a list.


In [ ]:
def answer(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    resp = llm.invoke(prompt.format_messages(context=context, question=question))
    pages = sorted({d.metadata.get("page", "?") for d in docs})
    return resp.content.strip(), pages

for q in [
    "How many casual leave days are provided each year?",
    "What is the reimbursement limit for internet at home?",
    "Who won the 2019 cricket world cup?",   # not in the doc -> should refuse
]:
    text, pages = answer(q)
    print("Q:", q)
    print("A:", text)
    print("pages:", pages)
    print("-" * 60)


### Optional: interactive input loop
Run this cell and type questions; enter `exit` to stop.


In [ ]:
while True:
    q = input("Question> ").strip()
    if not q:
        continue
    if q.lower() == "exit":
        break
    text, pages = answer(q)
    print("A:", text, "\npages:", pages)


## Optional - grade your task code with `grader.py`

Paste each task's code as a string and score it against the same rubric the Streamlit app uses.


In [ ]:
from grader import grade_task, assemble_source

# put each task's code in its own string (here: the reference solutions)
t1 = (
    "from langchain_community.document_loaders import PyPDFLoader\n"
    "loader = PyPDFLoader(SAMPLE_PDF)\n"
    "documents = loader.load()\n"
    "print('pages', len(documents))\n"
)
t2 = (
    "from langchain_text_splitters import RecursiveCharacterTextSplitter\n"
    "chunk_size = 1000\n"
    "chunk_overlap = 200\n"
    "splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)\n"
    "chunks = splitter.split_documents(documents)\n"
    "print('chunks', len(chunks))\n"
)

task_codes = [t1, t2]        # add t3, t4, t5 as you write them
tid = len(task_codes)        # grade the last one
src = assemble_source(task_codes, MODE)
res = grade_task(tid, task_codes[-1], lab_id="01-rag", accumulated_source=src, mode=MODE)
print("score", res["score"], "/", res["max_score"])
for c in res["criteria"]:
    print("  OK  " if c["passed"] else "  --  ", c["name"], "-", c["reason"])
